In [3]:
from main import main

NameError: name '_C' is not defined

In [ ]:
main('config_files/test.yaml')

## Dataset

In [3]:
import torch
from torch import nn, Tensor
from torch_geometric.data import Data, Dataset
import random

In [6]:
class CustomGraphDataset(Dataset):
    def __init__(self, num_graphs, max_nodes, max_edges, nr_node_features):
        self.num_graphs = num_graphs
        self.max_nodes = max_nodes
        self.max_edges = max_edges
        self.nr_node_features = nr_node_features
        self.data_list = []

        for _ in range(num_graphs):
            num_nodes = random.randint(1, max_nodes)
            num_edges = random.randint(0, min(num_nodes * (num_nodes - 1) // 2, max_edges))
            
            edge_index = torch.zeros((2, num_edges), dtype=torch.long)
            for i in range(num_edges):
                edge_index[0, i] = random.randint(0, num_nodes - 1)
                edge_index[1, i] = random.randint(0, num_nodes - 1)
            
            x = torch.rand((num_nodes, nr_node_features), dtype=torch.float)  # Node features (random for example)
            y = torch.tensor([random.randint(0, 4)], dtype=torch.long)  # Graph label (binary for example)

            data = Data(x=x, edge_index=edge_index, y=y)
            self.data_list.append(data)

    def __len__(self):
        return self.num_graphs

    def __getitem__(self, idx):
        return self.data_list[idx]

In [7]:
num_graphs = 10
max_nodes_per_graph = 50
max_edges_per_graph = 30
nr_node_features = 20

custom_dataset = CustomGraphDataset(num_graphs, max_nodes_per_graph, max_edges_per_graph, nr_node_features)

for i in range(num_graphs):
    data = custom_dataset[i]
    print(f"Graph {i + 1} - Nodes: {data.num_nodes}, Edges: {data.num_edges}, Label: {data.y.item()}")

Graph 1 - Nodes: 47, Edges: 7, Label: 2
Graph 2 - Nodes: 11, Edges: 29, Label: 1
Graph 3 - Nodes: 4, Edges: 5, Label: 4
Graph 4 - Nodes: 47, Edges: 25, Label: 3
Graph 5 - Nodes: 46, Edges: 17, Label: 4
Graph 6 - Nodes: 30, Edges: 7, Label: 4
Graph 7 - Nodes: 40, Edges: 24, Label: 1
Graph 8 - Nodes: 41, Edges: 7, Label: 3
Graph 9 - Nodes: 8, Edges: 4, Label: 0
Graph 10 - Nodes: 4, Edges: 3, Label: 1


## Dataloader 

In [8]:
from torch_geometric.loader import DataLoader

In [11]:
BATCH_SIZE = 5
train_loader = DataLoader(custom_dataset.data_list, batch_size=BATCH_SIZE) # merges batch_size graphs from a PyG dataset 

In [12]:
for idx, batch in enumerate(train_loader):
    print(batch)

DataBatch(x=[155, 20], edge_index=[2, 83], y=[5], batch=[155], ptr=[6])
DataBatch(x=[123, 20], edge_index=[2, 45], y=[5], batch=[123], ptr=[6])


## Config file

In [13]:
import yaml

In [41]:
cfg_path = 'config_files/test.yaml'

In [42]:
with open(cfg_path) as stream:
    cfg_dict = yaml.safe_load(stream)

In [43]:
cfg_dict

{'out_dir': 'results',
 'dataloader': {'data_path': '', 'radius': 30},
 'gnn': {'gnn_type': 'GCN',
  'num_layers': 2,
  'num_features': 20,
  'hidden_dim': 16,
  'embed_dim': 8,
  'num_classes': 4},
 'transformer': {'d_model': 128,
  'n_heads': 4,
  'dim_feedforward': 512,
  'dropout': 0.3,
  'num_layers': 4,
  'activation_func': 'relu',
  'num_encoder_layers': 4}}

## GCN

In [36]:
from torch.nn import Linear
from torch_geometric.nn import GCNConv, MessagePassing
import torch.nn.functional as F

class GCN(torch.nn.Module):
    def __init__(self, 
                 cfg,
                 dp_rate = 0.1):
        super().__init__()      
        #dp_rate = cfg['dp_rate'] if cfg['dp_rate'] is not None else dp_rate

        in_dim, hidden_dim = cfg['num_features'], cfg['hidden_dim']
        layers = []
        for l_idx in range(cfg['num_layers'] - 1):
            layers += [
                GCNConv(in_channels=in_dim, out_channels=hidden_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(dp_rate)
            ]
            in_dim = hidden_dim
        
        layers += [GCNConv(in_channels=in_dim, out_channels=cfg['embed_dim'])]
        self.layers = nn.ModuleList(layers)
        self.out = Linear(cfg['embed_dim'], cfg['num_classes'])

    def forward(self, x, edge_index):
        """
        Input:
            x: Adjacency matrix (n x obs)
            edge_index: gene expressiong (var x obs)
        """
        for layer in self.layers:
            if isinstance(layer, MessagePassing):
                x = layer(x, edge_index)
            else:
                x = layer(x)
        h = F.relu(x.mean(dim=0))
        z = self.out(h.unsqueeze(0))
        return x, z


In [103]:
model_GCN = GCN(cfg_dict['gnn'])
N_EPOCHS = 20
lr = 0.1
weight_decay = 5e-4
optimizer = torch.optim.Adam(model_GCN.parameters(), lr=lr, weight_decay=weight_decay)

In [104]:
model_GCN

GCN(
  (layers): ModuleList(
    (0): GCNConv(20, 16)
    (1): ReLU(inplace=True)
    (2): Dropout(p=0.1, inplace=False)
    (3): GCNConv(16, 8)
  )
  (out): Linear(in_features=8, out_features=4, bias=True)
)

In [88]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_GCN.parameters(), lr=0.02)

# Calculate accuracy
def accuracy(pred_y, y):
    return (pred_y == y).sum() / len(y)

# Data for animations
embeddings = []
losses = []
accuracies = []
outputs = []

# Training loop
for epoch in range(N_EPOCHS):
    for idx, batch in enumerate(train_loader): 
        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        h, z = model_GCN(batch.x, batch.edge_index)

        # Calculate loss function
        loss = criterion(z, data.y)

        # Calculate accuracy
        acc = accuracy(z.argmax(dim=1), data.y)

        # Compute gradients
        loss.backward()

        # Tune parameters
        optimizer.step()

        # Store data for animations
        embeddings.append(h)
        losses.append(loss)
        accuracies.append(acc)
        outputs.append(z.argmax(dim=1))

        # Print metrics every 10 epochs
        if epoch % 10 == 0:
            print(f'Epoch {epoch:>3} | Loss: {loss:.2f} | Acc: {acc*100:.2f}%')
            print(h.shape, z.shape)

Epoch   0 | Loss: 0.00 | Acc: 100.00%
torch.Size([155, 8]) torch.Size([1, 4])
Epoch   0 | Loss: 0.00 | Acc: 100.00%
torch.Size([123, 8]) torch.Size([1, 4])
Epoch  10 | Loss: 0.00 | Acc: 100.00%
torch.Size([155, 8]) torch.Size([1, 4])
Epoch  10 | Loss: 0.00 | Acc: 100.00%
torch.Size([123, 8]) torch.Size([1, 4])


## Transformer 

In [98]:
class TransformerNodeEncoder(nn.Module):
    """
    Sequence of: Dropout → Layer Norm → FC → nonlinearity → Dropout → FC → Dropout → Layer Norm + residual connections
    """
    
    def __init__(self, cfg):

        super().__init__()

        self.model_type = 'TransformerEncoder'

        self.gnn2transformer = nn.Linear(cfg['gnn']['embed_dim'], cfg['transformer']['d_model'])
        self.norm_input = nn.LayerNorm(cfg['transformer']['d_model'])
        self.cls_embedding = nn.Parameter(torch.randn([1, 1, cfg['transformer']['d_model']], requires_grad = True))
        
        encoder_layer = nn.TransformerEncoderLayer(
            cfg['transformer']['d_model'], cfg['transformer']['n_heads'], cfg['transformer']['dim_feedforward'], cfg['transformer']['dropout'], cfg['transformer']['activation_func']
        )
        encoder_norm = nn.LayerNorm(cfg['transformer']['d_model'])
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, cfg['transformer']['num_encoder_layers'], norm=encoder_norm)
        

    def forward(self, src, src_mask: Tensor = None):
        """
        Input: 
            src: final per-node GNN encodings ``[seq_len x embed_dim_gnn]``
            src_mask: final per-node GNN encodings ``[var/genes x var/genes]``
        """
        print(src.shape)
        #padded_h_node = torch.cat([, cls_embedding], dim=0)
        padded_h_node = self.gnn2transformer(src)
        print(padded_h_node.shape)
        padded_h_node = self.norm_input(padded_h_node)
        padded_h_node = padded_h_node.unsqueeze(1) # from [embed_dim_gnn, embed_dim_transformer] to [embed_dim_gnn, batch_size=1, embed_dim_transformer]
        print(padded_h_node.shape)
        if src_mask is None:
            # masked positions are filled with ('-inf') and unmasked positions filled with float (0.0)
            src_mask = nn.Transformer.generate_square_subsequent_mask(len(src))
        print(padded_h_node.shape, src_mask.shape)
        transformer_out = self.transformer_encoder(padded_h_node, src_mask)  # (S, B, h_d)
        return transformer_out

In [101]:
transformer_model = TransformerNodeEncoder(cfg_dict)
N_EPOCHS = 20
lr = 0.1
weight_decay = 5e-4
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=lr, weight_decay=weight_decay)

In [102]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=0.02)

# Calculate accuracy
def accuracy(pred_y, y):
    return (pred_y == y).sum() / len(y)

# Data for animations
embeddings = []
losses = []
accuracies = []
outputs = []

# Training loop
for epoch in range(N_EPOCHS):
    for idx, batch in enumerate(train_loader): 
        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        out = transformer_model(h, None)

        print(out.shape, out)

        # Calculate loss function
        loss = criterion(z, data.y)

        # Calculate accuracy
        acc = accuracy(z.argmax(dim=1), data.y)

        # Compute gradients
        loss.backward()

        # Tune parameters
        optimizer.step()

        # Store data for animations
        embeddings.append(h)
        losses.append(loss)
        accuracies.append(acc)
        outputs.append(z.argmax(dim=1))

        # Print metrics every 10 epochs
        if epoch % 10 == 0:
            print(f'Epoch {epoch:>3} | Loss: {loss:.2f} | Acc: {acc*100:.2f}%')

torch.Size([123, 8])
torch.Size([123, 128])
torch.Size([123, 1, 128])
torch.Size([123, 1, 128]) torch.Size([123, 123])
torch.Size([123, 1, 128]) tensor([[[-0.0384,  1.0897,  0.1734,  ..., -0.2738,  0.1366,  0.6347]],

        [[-0.3331,  1.0477, -2.0582,  ...,  0.7769, -0.0943,  0.0992]],

        [[-1.2638,  1.7449, -3.4022,  ...,  0.5640,  0.6374,  0.2166]],

        ...,

        [[ 0.1281,  1.3587, -1.8128,  ...,  1.2309,  0.2650,  0.6551]],

        [[-0.4156,  1.1201, -1.4031,  ...,  0.9088,  0.4756,  1.4148]],

        [[ 0.2740,  0.4644, -1.3885,  ...,  0.6041,  0.4581,  0.9701]]],
       grad_fn=<NativeLayerNormBackward0>)


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

## GNN + Transformer

https://github.com/ucbrise/graphtrans/blob/main/models/gnn_transformer.py

In [105]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_GCN.parameters(), lr=0.02)

# Calculate accuracy
def accuracy(pred_y, y):
    return (pred_y == y).sum() / len(y)

# Data for animations
embeddings = []
losses = []
accuracies = []
outputs = []

# Training loop
for epoch in range(N_EPOCHS):
    for idx, batch in enumerate(train_loader): 
        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        h, z = model_GCN(batch.x, batch.edge_index)
        out = transformer_model(h, None)

        print(out.shape)

        # cls = retrieve CLS embedding for graph level prediction

        # Calculate loss function
        loss = criterion(out, batch.y)

        # Calculate accuracy
        acc = accuracy(z.argmax(dim=1), data.y)

        # Compute gradients
        loss.backward()

        # Tune parameters
        optimizer.step()

        # Store data for animations
        embeddings.append(h)
        losses.append(loss)
        accuracies.append(acc)
        outputs.append(z.argmax(dim=1))

        # Print metrics every 10 epochs
        if epoch % 10 == 0:
            print(f'Epoch {epoch:>3} | Loss: {loss:.2f} | Acc: {acc*100:.2f}%')
            print(h.shape, z.shape)

torch.Size([155, 8])
torch.Size([155, 128])
torch.Size([155, 1, 128])
torch.Size([155, 1, 128]) torch.Size([155, 155])
torch.Size([155, 1, 128])


ValueError: Expected input batch_size (155) to match target batch_size (5).